# Progetto Data Science – Qualità dell'Aria a Milano (2021–2024)

**Obiettivo:** Raccogliere, pulire e analizzare dati ambientali di Milano, poi costruire un modello predittivo che stima i livelli di PM2.5 fino al **2028**.

**Fonti dati:**
1. **Open-Meteo Air Quality API** – PM2.5, PM10, NO2, O3 (modello riallinea ERA5, gratuito)
2. **Open-Meteo Archive API** – temperatura, umidità, vento, precipitazioni (ERA5, gratuito)
3. **Open-Meteo Climate API (CMIP6)** – proiezioni climatiche giornaliere 2021–2028 (ensemble di 3 modelli GCM)

**Coordinate Milano:** lat 45.4654, lon 9.1859

---

In [ ]:
# ── Dipendenze (decommentare e runnare una volta se mancano) ──────────────────
# !pip install requests pandas numpy matplotlib seaborn scikit-learn statsmodels

In [ ]:
# ── Import e configurazione globale ──────────────────────────────────────────
import os, time, warnings
warnings.filterwarnings('ignore')

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titleweight'] = 'bold'

# Coordinate e periodo
LAT, LON = 45.4654, 9.1859
START, END = '2021-01-01', '2024-12-31'

RAW_DIR  = os.path.join('data', 'raw')
PROC_DIR = os.path.join('data', 'processed')
os.makedirs(RAW_DIR,  exist_ok=True)
os.makedirs(PROC_DIR, exist_ok=True)

print('Setup completato.')
print(f'Periodo: {START} → {END} | Milano ({LAT}°N, {LON}°E)')

---
## Task 1 – Raccolta dei Dati

Scarichiamo dati da tre fonti distinte e li salviamo in `data/raw/` come CSV.

### 1.1 – Qualità dell'aria (Open-Meteo Air Quality)

API: `https://air-quality-api.open-meteo.com/v1/air-quality`  
Gratuita, senza chiave. Dati orari di PM2.5, PM10, NO2 e O3 derivati dal modello CAMS.

In [ ]:
def fetch_qualita_aria(lat, lon, start_date, end_date):
    """
    Scarica dati orari di qualità dell'aria da Open-Meteo Air Quality.

    Parametri
    ---------
    lat, lon     : float – coordinate geografiche
    start_date   : str   – 'YYYY-MM-DD' inizio periodo
    end_date     : str   – 'YYYY-MM-DD' fine periodo

    Ritorna
    -------
    pd.DataFrame con colonne: datetime, pm2_5, pm10, nitrogen_dioxide, ozone
    """
    url = 'https://air-quality-api.open-meteo.com/v1/air-quality'
    params = {
        'latitude':   lat,
        'longitude':  lon,
        'hourly':     'pm2_5,pm10,nitrogen_dioxide,ozone',
        'start_date': start_date,
        'end_date':   end_date,
        'timezone':   'UTC',
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    h = r.json()['hourly']
    return pd.DataFrame({
        'datetime':         pd.to_datetime(h['time']),
        'pm2_5':            h['pm2_5'],
        'pm10':             h['pm10'],
        'nitrogen_dioxide': h['nitrogen_dioxide'],
        'ozone':            h['ozone'],
    })


print('Scaricamento qualità dell\'aria…')
df_aria = fetch_qualita_aria(LAT, LON, START, END)

path_aria = os.path.join(RAW_DIR, 'aria_milano.csv')
df_aria.to_csv(path_aria, index=False)
print(f'Salvato: {len(df_aria):,} righe → {path_aria}')
df_aria.head()

### 1.2 – Dati meteorologici (Open-Meteo Archive)

API: `https://archive-api.open-meteo.com/v1/archive`  
Rianalisi ERA5 oraria: temperatura a 2 m, umidità relativa, velocità del vento a 10 m, precipitazioni.

In [ ]:
def fetch_meteo(lat, lon, start_date, end_date):
    """
    Scarica dati meteo orari storici da Open-Meteo Archive (rianalisi ERA5).

    Ritorna
    -------
    pd.DataFrame con colonne: datetime, temperatura, umidita, vento_kmh, precipitazioni
    """
    url = 'https://archive-api.open-meteo.com/v1/archive'
    params = {
        'latitude':           lat,
        'longitude':          lon,
        'start_date':         start_date,
        'end_date':           end_date,
        'hourly':             'temperature_2m,relativehumidity_2m,windspeed_10m,precipitation',
        'timezone':           'UTC',
        'temperature_unit':   'celsius',
        'windspeed_unit':     'kmh',
        'precipitation_unit': 'mm',
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    h = r.json()['hourly']
    return pd.DataFrame({
        'datetime':      pd.to_datetime(h['time']),
        'temperatura':   h['temperature_2m'],
        'umidita':       h['relativehumidity_2m'],
        'vento_kmh':     h['windspeed_10m'],
        'precipitazioni':h['precipitation'],
    })


print('Scaricamento dati meteo…')
df_meteo = fetch_meteo(LAT, LON, START, END)

path_meteo = os.path.join(RAW_DIR, 'meteo_milano.csv')
df_meteo.to_csv(path_meteo, index=False)
print(f'Salvato: {len(df_meteo):,} righe → {path_meteo}')
df_meteo.head()

### 1.3 – Proiezioni climatiche 2021–2028 (Open-Meteo Climate API – CMIP6)

API: `https://climate-api.open-meteo.com/v1/climate`  
Gratuita, senza chiave. Proiezioni giornaliere da modelli CMIP6 (ensemble di 3 GCM).  
Questo dataset copre sia il **passato (2021–2024)** che il **futuro (2025–2028)** e sarà usato
nel Task 4 per contestualizzare e migliorare la previsione al 2028.

In [ ]:
def fetch_proiezioni_cmip6(lat, lon, start_date, end_date):
    """
    Scarica proiezioni climatiche giornaliere CMIP6 da Open-Meteo Climate API.

    Modelli usati:
      - CMCC_CM2_VHR4: modello ad alta risoluzione del Centro Euro-Mediterraneo (IT)
      - MPI_ESM1_2_XR: modello ad alta risoluzione Max-Planck-Institut (DE)
      - EC_Earth3P_HR: modello europeo EC-Earth (NL/IE/SE)

    L'ensemble mean riduce l'incertezza del singolo modello.

    Ritorna
    -------
    pd.DataFrame con colonne: data, temp_max_ensemble, precip_ensemble
    """
    url = 'https://climate-api.open-meteo.com/v1/climate'
    modelli = 'CMCC_CM2_VHR4,MPI_ESM1_2_XR,EC_Earth3P_HR'
    params = {
        'latitude':   lat,
        'longitude':  lon,
        'start_date': start_date,
        'end_date':   end_date,
        'daily':      'temperature_2m_max,precipitation_sum',
        'models':     modelli,
    }
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()
    d = r.json()['daily']
    df = pd.DataFrame({'data': pd.to_datetime(d['time'])})

    # Calcola ensemble mean per temperatura massima
    cols_temp = [k for k in d if k.startswith('temperature_2m_max_')]
    cols_prec = [k for k in d if k.startswith('precipitation_sum_')]
    for c in cols_temp + cols_prec:
        df[c] = d[c]
    df['temp_max_ensemble']  = df[cols_temp].mean(axis=1)
    df['precip_ensemble']    = df[cols_prec].mean(axis=1)

    return df[['data', 'temp_max_ensemble', 'precip_ensemble']]


print('Scaricamento proiezioni CMIP6 (2021–2028)…')
df_clima = fetch_proiezioni_cmip6(LAT, LON, start_date='2021-01-01', end_date='2028-12-31')

path_clima = os.path.join(RAW_DIR, 'clima_cmip6_milano.csv')
df_clima.to_csv(path_clima, index=False)
print(f'Salvato: {len(df_clima):,} giorni → {path_clima}')
df_clima.head()

### 1.4 – Riepilogo raccolta

In [ ]:
# ── Verifica rapida su tutti e tre i dataset ──────────────────────────────────
for nome, df, col_dt in [
    ('aria_milano.csv',        df_aria,  'datetime'),
    ('meteo_milano.csv',       df_meteo, 'datetime'),
    ('clima_cmip6_milano.csv', df_clima, 'data'),
]:
    print(f'── {nome} ──')
    print(f'  Shape   : {df.shape}')
    print(f'  Colonne : {list(df.columns)}')
    print(f'  Periodo : {df[col_dt].min()} → {df[col_dt].max()}')
    mancanti = df.isnull().sum()
    mancanti = mancanti[mancanti > 0]
    print(f'  Mancanti: {mancanti.to_dict() if not mancanti.empty else "nessuno"}')
    print()

---
## Task 2 – Pulizia e Integrazione dei Dati

Uniamo i dataset orari (aria + meteo) su `datetime`, aggiungiamo feature temporali
e salviamo il dataset unificato in `data/processed/`.

In [ ]:
# ── 2.1 Carica i CSV raw ──────────────────────────────────────────────────────
df_aria  = pd.read_csv(os.path.join(RAW_DIR, 'aria_milano.csv'),  parse_dates=['datetime'])
df_meteo = pd.read_csv(os.path.join(RAW_DIR, 'meteo_milano.csv'), parse_dates=['datetime'])

# ── 2.2 Merge inner su datetime (solo ore presenti in entrambi) ───────────────
df = df_aria.merge(df_meteo, on='datetime', how='inner')

# ── 2.3 Pulizia: rimuovi infiniti e imputa NaN con mediana della colonna ─────
df = df.replace([np.inf, -np.inf], np.nan)
for col in df.select_dtypes(include=[np.number]).columns:
    df[col] = df[col].fillna(df[col].median())

righe_prima = len(df)
df.dropna(inplace=True)
print(f'Righe dopo pulizia: {len(df):,} (rimosse: {righe_prima - len(df)})')

# ── 2.4 Feature temporali ─────────────────────────────────────────────────────
df['ora']           = df['datetime'].dt.hour
df['giorno_sett']   = df['datetime'].dt.dayofweek   # 0=lunedì, 6=domenica
df['mese']          = df['datetime'].dt.month
df['anno']          = df['datetime'].dt.year
df['is_weekend']    = df['giorno_sett'].isin([5, 6]).astype(int)

# ── 2.5 Salva dataset unificato ───────────────────────────────────────────────
out = os.path.join(PROC_DIR, 'milano_unificato.csv')
df.to_csv(out, index=False)

print(f'\nSalvato → {out}')
print(f'Shape: {df.shape}')
print(f'Colonne: {list(df.columns)}')
df.head()

In [ ]:
# ── 2.6 Statistiche descrittive ───────────────────────────────────────────────
print('=== STATISTICHE DESCRITTIVE ===')
df.describe().round(2)

---
## Task 3 – Analisi Esplorativa (EDA)

Indaghiamo tre domande:
1. Come si evolvono gli inquinanti nel tempo?
2. Quali variabili meteorologiche correlano con la qualità dell'aria?
3. Esistono pattern orari o stagionali nel PM2.5?

In [ ]:
# ── Ricarica dataset (utile in caso di restart del kernel) ───────────────────
df = pd.read_csv(os.path.join(PROC_DIR, 'milano_unificato.csv'), parse_dates=['datetime'])
print(f'Dataset caricato: {df.shape[0]:,} righe, {df.shape[1]} colonne')

In [ ]:
# ── Grafico 1: Serie storica PM2.5 con media mobile 24h ──────────────────────
fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(df['datetime'], df['pm2_5'],
        color='#b0c4de', linewidth=0.5, alpha=0.7, label='PM2.5 orario')

rolling = df.set_index('datetime')['pm2_5'].rolling('24h').mean()
ax.plot(rolling.index, rolling.values,
        color='#1f4e79', linewidth=2, label='Media mobile 24h')

ax.axhline(10, color='orange', linestyle='--', linewidth=1.2, label='Limite OMS (10 µg/m³)')
ax.axhline(25, color='red',    linestyle='--', linewidth=1.2, label='Limite UE (25 µg/m³)')

ax.set_title('Concentrazione PM2.5 – Milano (2021–2024)')
ax.set_xlabel('Data')
ax.set_ylabel('PM2.5 (µg/m³)')
ax.set_xlim(df['datetime'].min(), df['datetime'].max())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(PROC_DIR, 'mi_grafico1_pm25.png'), dpi=150)
plt.show()

In [ ]:
# ── Grafico 2: Tutti gli inquinanti (medie giornaliere) ───────────────────────
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

inquinanti = [
    ('pm2_5',            'PM2.5 (µg/m³)', '#1f77b4'),
    ('pm10',             'PM10 (µg/m³)',  '#ff7f0e'),
    ('nitrogen_dioxide', 'NO2 (µg/m³)',   '#2ca02c'),
    ('ozone',            'O3 (µg/m³)',    '#9467bd'),
]
for ax, (col, etichetta, colore) in zip(axes, inquinanti):
    giornaliero = df.set_index('datetime')[col].resample('D').mean()
    ax.fill_between(giornaliero.index, giornaliero.values, alpha=0.3, color=colore)
    ax.plot(giornaliero.index, giornaliero.values, color=colore, linewidth=1.5)
    ax.set_ylabel(etichetta)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
axes[0].set_title('Media Giornaliera degli Inquinanti – Milano (2021–2024)')
plt.tight_layout()
plt.savefig(os.path.join(PROC_DIR, 'mi_grafico2_inquinanti.png'), dpi=150)
plt.show()

In [ ]:
# ── Grafico 3: Heatmap di correlazione ───────────────────────────────────────
variabili = ['pm2_5', 'pm10', 'nitrogen_dioxide', 'ozone',
             'temperatura', 'umidita', 'vento_kmh', 'precipitazioni']
corr = df[variabili].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5,
            ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Matrice di Correlazione – Variabili Ambientali Milano')
plt.tight_layout()
plt.savefig(os.path.join(PROC_DIR, 'mi_grafico3_correlazione.png'), dpi=150)
plt.show()

In [ ]:
# ── Grafico 4: Pattern orario e stagionale del PM2.5 ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sinistra: PM2.5 medio per ora del giorno
per_ora = df.groupby('ora')['pm2_5'].mean()
axes[0].bar(per_ora.index, per_ora.values, color='#1f4e79', alpha=0.85, edgecolor='white')
axes[0].set_title('PM2.5 Medio per Ora del Giorno')
axes[0].set_xlabel('Ora (0 = mezzanotte)')
axes[0].set_ylabel('PM2.5 (µg/m³)')
axes[0].set_xticks(range(0, 24, 2))

# Destra: PM2.5 medio per mese (stagionalità)
mesi = ['Gen','Feb','Mar','Apr','Mag','Giu','Lug','Ago','Set','Ott','Nov','Dic']
per_mese = df.groupby('mese')['pm2_5'].mean()
colori_mese = ['#4c78a8'] * 3 + ['#72b7b2'] * 6 + ['#4c78a8'] * 3
axes[1].bar([mesi[m-1] for m in per_mese.index], per_mese.values,
            color=colori_mese, alpha=0.85, edgecolor='white')
axes[1].set_title('PM2.5 Medio per Mese (Stagionalità)')
axes[1].set_xlabel('Mese')
axes[1].set_ylabel('PM2.5 (µg/m³)')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Pattern Temporali della Qualità dell\'Aria a Milano',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PROC_DIR, 'mi_grafico4_pattern.png'), dpi=150)
plt.show()

In [ ]:
# ── Grafico 5: Temperatura vs PM2.5 (colorato per stagione) ──────────────────
fig, ax = plt.subplots(figsize=(9, 6))

# Campiona 5 000 punti per leggibilità
sample = df.sample(min(5000, len(df)), random_state=42)
scatter = ax.scatter(
    sample['temperatura'], sample['pm2_5'],
    c=sample['mese'], cmap='RdYlBu_r',
    alpha=0.3, s=6, edgecolors='none'
)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Mese')
cbar.set_ticks([1, 3, 6, 9, 12])
cbar.set_ticklabels(['Gen', 'Mar', 'Giu', 'Set', 'Dic'])

ax.set_title('Temperatura vs PM2.5 a Milano\n(colorato per mese)')
ax.set_xlabel('Temperatura (°C)')
ax.set_ylabel('PM2.5 (µg/m³)')
plt.tight_layout()
plt.savefig(os.path.join(PROC_DIR, 'mi_grafico5_temp_pm25.png'), dpi=150)
plt.show()

corr_val = df[['temperatura', 'pm2_5']].corr().iloc[0, 1]
print(f'Correlazione temperatura–PM2.5: {corr_val:.3f}')
print('→ Correlazione negativa attesa: inverno freddo = aria ferma = più polveri fini.')

### Osservazioni chiave dall'EDA

- **PM2.5 invernale** (Dic–Feb) supera regolarmente il limite OMS di 10 µg/m³,
  con picchi che raggiungono o superano il limite UE di 25 µg/m³.
- **Stagionalità marcata**: PM2.5 massimo in inverno (stagnazione atmosferica,
  riscaldamento, nebbia padana), minimo in estate (vento, piogge convettive).
- **Correlazione negativa con la temperatura** (r ≈ −0.35): freddo → inversione termica
  → accumulo di polveri in strati bassi.
- **Vento**: correlazione negativa con tutti gli inquinanti — il vento disperde
  le particelle. Milano è nota per la bassa ventosità della Pianura Padana.
- **O3 opposto**: l'ozono è più alto in estate (radiazione UV + caldo), anticorrelato con PM2.5.

---
## Task 4 – Modello Predittivo al 2028 (SARIMA)

Usiamo **SARIMA** (Seasonal ARIMA) per prevedere il PM2.5 giornaliero di Milano fino a fine 2028.

**Struttura del modello:**
- Lavoriamo su **medie giornaliere** di PM2.5 (risampling da orario a giornaliero)
- `SARIMA(1,1,1)(1,1,1,7)` cattura trend, differenziazione e stagionalità settimanale
- **Split**: train 2021–2023 (3 anni) | test 2024 (1 anno di validazione)
- Dopo la validazione, addestriamo sul dataset completo 2021–2024 e prevediamo 2025–2028

**Nota sulla previsione a lungo termine:**  
Con 4 anni di training, una previsione a 4 anni ha incertezza crescente.
Le bande di confidenza si allargano progressivamente — questo è corretto e atteso.

In [ ]:
# ── Prepara serie temporale giornaliera PM2.5 ─────────────────────────────────
df = pd.read_csv(os.path.join(PROC_DIR, 'milano_unificato.csv'), parse_dates=['datetime'])

ts = (
    df.set_index('datetime')['pm2_5']
      .resample('D').mean()
      .rename('pm2_5')
)
ts = ts.interpolate(method='time')  # colma eventuali giorni mancanti

print(f'Serie temporale: {len(ts)} giorni')
print(f'Periodo       : {ts.index[0].date()} → {ts.index[-1].date()}')
print(f'PM2.5 medio   : {ts.mean():.2f} µg/m³')
print(f'PM2.5 max     : {ts.max():.2f} µg/m³')

# Grafico serie giornaliera
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(ts.index, ts.values, color='#1f4e79', linewidth=1.5)
ax.axhline(10, color='orange', linestyle='--', linewidth=1, label='Limite OMS')
ax.axhline(25, color='red',    linestyle='--', linewidth=1, label='Limite UE')
ax.set_title('PM2.5 Giornaliero – Milano (2021–2024)')
ax.set_xlabel('Data')
ax.set_ylabel('PM2.5 (µg/m³)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PROC_DIR, 'mi_grafico6_pm25_giorn.png'), dpi=150)
plt.show()

In [ ]:
# ── Split train/test: train 2021-2023, test 2024 ─────────────────────────────
train = ts[ts.index.year < 2024]
test  = ts[ts.index.year == 2024]

print(f'Train: {len(train)} giorni ({train.index[0].date()} → {train.index[-1].date()})')
print(f'Test : {len(test)} giorni ({test.index[0].date()} → {test.index[-1].date()})')

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(train.index, train.values, color='#1f4e79', linewidth=1.5, label='Training (2021–2023)')
ax.plot(test.index,  test.values,  color='#e07b39', linewidth=1.5, label='Test (2024)')
ax.axvline(test.index[0], color='gray', linestyle='--', linewidth=1)
ax.set_title('Divisione Train / Test – PM2.5 Giornaliero Milano')
ax.set_xlabel('Data')
ax.set_ylabel('PM2.5 (µg/m³)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PROC_DIR, 'mi_grafico7_split.png'), dpi=150)
plt.show()

In [ ]:
# ── Addestramento SARIMA(1,1,1)(1,1,1,7) ────────────────────────────────────
# (1,1,1): AR=1, differenziazione=1, MA=1
# (1,1,1,7): componente stagionale settimanale
print('Addestramento SARIMA sul set di training (2021–2023)…')
print('Questo può richiedere ~1-2 minuti.')

modello = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)
risultato = modello.fit(disp=False)

print(risultato.summary().tables[0])
print(f'\nAIC: {risultato.aic:.1f}  |  BIC: {risultato.bic:.1f}')

In [ ]:
# ── Previsione sul test set (2024) e valutazione ──────────────────────────────
previsione_test = risultato.get_forecast(steps=len(test))
pred_test       = previsione_test.predicted_mean
ci_test         = previsione_test.conf_int(alpha=0.05)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.index, train.values,
        color='#1f4e79', linewidth=1.2, label='Training (2021–2023)')
ax.plot(test.index, test.values,
        color='#e07b39', linewidth=2, label='Valori reali 2024')
ax.plot(pred_test.index, pred_test.values,
        color='#2ca02c', linewidth=2, linestyle='--', label='SARIMA – previsione 2024')
ax.fill_between(ci_test.index, ci_test.iloc[:, 0], ci_test.iloc[:, 1],
                color='#2ca02c', alpha=0.15, label='IC 95%')
ax.axvline(test.index[0], color='gray', linestyle=':', linewidth=1)
ax.set_title('SARIMA – Previsione vs Reale (2024) | Milano')
ax.set_xlabel('Data')
ax.set_ylabel('PM2.5 (µg/m³)')
ax.set_ylim(0, max(ts.max(), pred_test.max()) * 1.15)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PROC_DIR, 'mi_grafico8_sarima_test.png'), dpi=150)
plt.show()

In [ ]:
# ── Metriche di valutazione ───────────────────────────────────────────────────
rmse = np.sqrt(mean_squared_error(test.values, pred_test.values))
mae  = mean_absolute_error(test.values, pred_test.values)
mape = np.mean(np.abs((test.values - pred_test.values) /
               np.where(test.values == 0, 1, test.values))) * 100
baseline_rmse = np.sqrt(mean_squared_error(test.values, [train.mean()] * len(test)))

print('=== VALUTAZIONE SUL TEST SET (2024) ===')
print(f'RMSE     : {rmse:.2f} µg/m³')
print(f'MAE      : {mae:.2f} µg/m³')
print(f'MAPE     : {mape:.1f}%')
print(f'Baseline : RMSE = {baseline_rmse:.2f} µg/m³  (predice sempre la media del training)')
print(f'Miglioramento SARIMA vs baseline: {((baseline_rmse - rmse) / baseline_rmse * 100):.1f}%')

In [ ]:
# ── Previsione 2025–2028: ri-addestramento su tutto 2021-2024 ─────────────────
print('Ri-addestramento SARIMA su tutto il dataset 2021–2024…')
print('(~2-3 minuti per 4 anni di dati giornalieri)')

modello_finale = SARIMAX(
    ts,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)
risultato_finale = modello_finale.fit(disp=False)

# Previsione 2025-2028 (4 anni = ~1461 giorni)
GIORNI_FORECAST = 365 * 4
forecast_obj = risultato_finale.get_forecast(steps=GIORNI_FORECAST)
forecast      = forecast_obj.predicted_mean
ci_forecast   = forecast_obj.conf_int(alpha=0.05)

print(f'\nPrevisione generata per {GIORNI_FORECAST} giorni')
print(f'Periodo previsto: {forecast.index[0].date()} → {forecast.index[-1].date()}')
print(f'PM2.5 medio previsto 2025–2028: {forecast.mean():.2f} µg/m³')

# Salva previsione come CSV
df_forecast = pd.DataFrame({
    'data':        forecast.index,
    'pm2_5_prev':  forecast.values,
    'ic_lower':    ci_forecast.iloc[:, 0].values,
    'ic_upper':    ci_forecast.iloc[:, 1].values,
})
df_forecast.to_csv(os.path.join(PROC_DIR, 'previsione_2025_2028.csv'), index=False)
print('Previsione salvata → previsione_2025_2028.csv')

In [ ]:
# ── Grafico finale: storico 2021-2024 + previsione 2025-2028 ─────────────────
fig, ax = plt.subplots(figsize=(16, 6))

# Dati storici
ax.plot(ts.index, ts.values,
        color='#1f4e79', linewidth=1.2, alpha=0.9, label='PM2.5 storico (2021–2024)')

# Previsione SARIMA
ax.plot(forecast.index, forecast.values,
        color='#2ca02c', linewidth=2, linestyle='--', label='Previsione SARIMA (2025–2028)')

# Banda di confidenza 95%
ax.fill_between(ci_forecast.index,
                ci_forecast.iloc[:, 0], ci_forecast.iloc[:, 1],
                color='#2ca02c', alpha=0.12, label='Intervallo di confidenza 95%')

# Linee di riferimento
ax.axhline(10, color='orange', linestyle=':', linewidth=1.2, label='Limite OMS (10 µg/m³)')
ax.axhline(25, color='red',    linestyle=':', linewidth=1.2, label='Limite UE (25 µg/m³)')
ax.axvline(pd.Timestamp('2025-01-01'), color='gray', linestyle='--', linewidth=1.5,
           label='Inizio previsione (gen 2025)')

ax.set_title('PM2.5 a Milano – Storico 2021–2024 e Previsione SARIMA 2025–2028',
             fontsize=13)
ax.set_xlabel('Anno')
ax.set_ylabel('PM2.5 (µg/m³)')
ax.set_ylim(0, max(ts.max(), ci_forecast.iloc[:, 1].max()) * 1.1)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(PROC_DIR, 'mi_grafico9_previsione2028.png'), dpi=150)
plt.show()

In [ ]:
# ── Stima PM2.5 medio per anno (2025-2028) ───────────────────────────────────
print('=== STIMA PM2.5 MEDIO PER ANNO (PREVISIONE) ===')
df_fc = df_forecast.copy()
df_fc['anno'] = pd.to_datetime(df_fc['data']).dt.year
per_anno = df_fc.groupby('anno').agg(
    pm2_5_medio=('pm2_5_prev', 'mean'),
    ic_lower=('ic_lower', 'mean'),
    ic_upper=('ic_upper', 'mean')
).round(2)
print(per_anno.to_string())

# Confronto con storico
print('\n=== STORICO PM2.5 MEDIO ANNUO ===')
storico_anno = ts.groupby(ts.index.year).mean().round(2)
print(storico_anno.to_string())

---
## Task 5 – Valutazione e Interpretazione

### Prestazioni del modello

| Metrica | Descrizione | Valore ideale |
|---------|-------------|---------------|
| **RMSE** | Errore quadratico medio (µg/m³) | Basso |
| **MAE**  | Errore assoluto medio (µg/m³) | Basso |
| **MAPE** | Errore percentuale medio | Basso |

Un MAPE elevato è atteso quando il PM2.5 si avvicina a zero (estate) —
gli errori percentuali si gonfiano artificialmente su valori piccoli.
Il confronto con la **baseline** (predire sempre la media del training)
mostra se SARIMA ha appreso qualcosa di utile.

In [ ]:
# ── Riepilogo metriche in tabella ─────────────────────────────────────────────
tabella = pd.DataFrame({
    'Modello': ['Baseline (media)', 'SARIMA(1,1,1)(1,1,1,7)'],
    'RMSE (µg/m³)': [
        round(baseline_rmse, 2),
        round(rmse, 2)
    ],
    'MAE (µg/m³)': [
        round(mean_absolute_error(test.values, [train.mean()] * len(test)), 2),
        round(mae, 2)
    ],
}).set_index('Modello')
print(tabella.to_string())

### Contestualizzazione con le proiezioni climatiche CMIP6

Il dataset **CMIP6** scaricato nel Task 1.3 fornisce proiezioni di temperatura fino al 2028.
La correlazione negativa rilevata nell'EDA (temperatura → PM2.5) suggerisce che:
- **Temperature più alte in estate** ridurranno i picchi invernali? Non necessariamente:
  gli inverni milanesi resteranno freddi — la Pianura Padana è strutturalmente a rischio
  di inversioni termiche indipendentemente dal riscaldamento globale.
- **Precipitazioni estive più intense** (clausius-clapeyron) potrebbero ridurre
  le concentrazioni estive di PM.

### Limiti del modello
- **SARIMA è univariato**: non usa temperatura, vento o mobilità come input.
  Un modello **SARIMAX con variabili esogene** (vento, inversioni termiche) migliorerebbe la precisione.
- **Orizzonte di 4 anni**: le bande di confidenza si allargano progressivamente —
  la previsione al 2028 va interpretata come **tendenza**, non come valore puntuale.
- **No politiche pubbliche**: il modello non include l'effetto di eventuali politiche
  (blocchi del traffico, transizione energetica, PNRR) che potrebbero ridurre il PM2.5.
- **Dati modellati (Open-Meteo)**: i dati di input sono dati di rianalisi, non misurazioni
  dirette da stazioni ARPA. Per un'analisi regolamentare si dovrebbero usare dati ARPA validati.

### Possibili miglioramenti
1. **SARIMAX**: aggiungere velocità del vento e temperatura come variabili esogene
2. **Prophet (Meta)**: gestisce automaticamente stagionalità multipla (settimanale + annuale)
   ed è più robusto per previsioni a lungo termine
3. **LSTM**: reti neurali ricorrenti per catturare dipendenze non lineari
4. **Dato ARPA**: sostituire Open-Meteo con misurazioni da stazioni certificate ARPA Lombardia
5. **Feature di mobilità**: integrare dati di traffico o rilevazioni dei contatori comunali